In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data=pd.read_csv("DateFruit_Dataset.csv")
X=data.drop("Class",axis=1)
y=data["Class"]

In [ ]:
y.unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [ ]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)


In [ ]:
scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

**ANN**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [ ]:
X_train_tensor=torch.tensor(X_train_scale,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test_scale,dtype=torch.float32)

Y_train_tensor=torch.tensor(Y_train_scale,dtype=torch.long)
Y_test_tensor=torch.tensor(Y_test_scale,dtype=torch.long)

In [ ]:
train_dataset=TensorDataset(X_train_tensor,Y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,Y_test_tensor)

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [ ]:
# Build our Model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            #hidden layer 01
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            #hidden Layer 02
            nn.Linear(64, 64),
            nn.ReLU(),

            #output layer
            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
model = ANN()

# loss & optim
criteria = nn.CrossEntropyLoss()  #for Multiclass Classificationn
optimizer = optim.Adam(model.parameters())

In [ ]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # params update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

RuntimeError: 0D or 1D target tensor expected, multi-target not supported

In [ ]:
# Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb.float()) # Ensure input is float
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actual samples in each batch

print("accuracy: ", correct/total * 100)

accuracy:  453.3333333333333
